# SCRIPT 4 — HYDROLOGIC AND FLOOD METRIC ASSESSMENT
## Brahmaputra Morphodynamics Study (AP-GARSS 2026)




In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio import features
from shapely.geometry import box, MultiPolygon, Polygon
from shapely.ops import unary_union
from shapely.geometry import mapping
import os
from google.colab import drive
drive.mount('/content/drive')





ROOT     = '/content/drive/MyDrive/BRAHMA_MORPHO'
PRED_DIR = f'{ROOT}/predictions'
VAL_DIR  = f'{ROOT}/validation'
SEG_DIR = f'{VAL_DIR}/Morph_Segments/'
INF_DIR  = f'{ROOT}/inference/scenes'
FIG_DIR  = f'{ROOT}/figures/script4'
os.makedirs(FIG_DIR,exist_ok=True)
os.makedirs(VAL_DIR,exist_ok=True)


Run once before processing individual years for the analysis

In [ ]:
# ── Section 4 Setup —  ──────────


# Pixel resolution and reach parameters
RES_M        = 10          # metres per pixel
RES_KM2      = (RES_M ** 2) / 1e6   # km² per pixel = 0.0001
REACH_LEN_KM = 131         # Tezpur to Guwahati centreline length

CLASS_NAMES  = {0:'Background', 1:'Active Channel',
                2:'Active Bar',  3:'Vegetated Bar'}
CLASS_COLORS = ['#1A0400','#66CCFF','#E6F2FF','#66FFB3']

# Load braidplain mask — static reference for all years
val_mask_path = sorted(glob.glob(
    os.path.join(INF_DIR, 'inference_2025_validation_mask*.tif')
))[0]
with rasterio.open(val_mask_path) as src:
    braidplain    = src.read(1) > 0
    bp_transform  = src.transform
    bp_crs        = src.crs

BRAIDPLAIN_AREA_KM2 = float(braidplain.sum()) * RES_KM2
BRAIDPLAIN_PX       = int(braidplain.sum())

print('Section 4 Setup Complete')
print(f'Braidplain area     : {BRAIDPLAIN_AREA_KM2:.2f} km²')
print(f'Braidplain pixels   : {BRAIDPLAIN_PX:,}')
print(f'Reach length        : {REACH_LEN_KM} km')
print(f'Pixel resolution    : {RES_M}m  ({RES_KM2} km² per pixel)')
print(f'Prediction folder   : {PRED_DIR}')
print(f'Braidplain mask from: {os.path.basename(val_mask_path)}')

# Storage for time series — fill as you process each year
annual_metrics = {}
print('\nReady. Process each year using the cell below.')
print('Years to process: 2019 2020 2021 2022 2023 2024 2025 2026')

Per Year Change Analysis

In [ ]:
# ── Process one year — change YEAR and run ────────────────────────────────
YEAR = '2026'    # ← CHANGE THIS for each year

pred_path = os.path.join(
    PRED_DIR, f'attention_unet_pred_{YEAR}_braidplain.tif'
)

if not os.path.exists(pred_path):
    print(f'ERROR: {pred_path} not found')
else:
    with rasterio.open(pred_path) as src:
        pred = src.read(1)   # (H, W) uint8

    # Align to braidplain mask if shapes differ
    if pred.shape != braidplain.shape:
        min_h = min(pred.shape[0], braidplain.shape[0])
        min_w = min(pred.shape[1], braidplain.shape[1])
        pred      = pred[:min_h,      :min_w]
        bp        = braidplain[:min_h, :min_w]
    else:
        bp = braidplain

    # Extract pixels inside braidplain only
    pred_bp = pred[bp]

    # Pixel counts
    n_channel  = int(np.sum(pred_bp == 1))
    n_bare_bar = int(np.sum(pred_bp == 2))
    n_veg_bar  = int(np.sum(pred_bp == 3))
    n_total_bp = int(bp.sum())
    n_total_bar= n_bare_bar + n_veg_bar

    # Areas in km²
    area_channel  = n_channel  * RES_KM2
    area_bare_bar = n_bare_bar * RES_KM2
    area_veg_bar  = n_veg_bar  * RES_KM2
    area_total_bar= n_total_bar* RES_KM2

    # Fractions relative to braidplain area
    frac_channel   = n_channel   / n_total_bp
    frac_total_bar = n_total_bar / n_total_bp
    frac_veg_bar   = n_veg_bar   / n_total_bp
    frac_bare_bar  = n_bare_bar  / n_total_bp

    # Vegetated fraction of total bar
    veg_of_bar = n_veg_bar / max(n_total_bar, 1)

    # Mean channel width
    # channel area / reach length
    mean_ch_width_km = area_channel / REACH_LEN_KM
    mean_ch_width_m  = mean_ch_width_km * 1000

    # Store for time series
    annual_metrics[YEAR] = {
        'year'              : int(YEAR),
        'channel_area_km2'  : round(area_channel,  2),
        'bare_bar_area_km2' : round(area_bare_bar, 2),
        'veg_bar_area_km2'  : round(area_veg_bar,  2),
        'total_bar_area_km2': round(area_total_bar, 2),
        'braidplain_area_km2': round(BRAIDPLAIN_AREA_KM2, 2),
        'frac_channel'      : round(frac_channel,   4),
        'frac_total_bar'    : round(frac_total_bar, 4),
        'frac_veg_bar'      : round(frac_veg_bar,   4),
        'frac_bare_bar'     : round(frac_bare_bar,  4),
        'veg_of_bar'        : round(veg_of_bar,     4),
        'mean_ch_width_m'   : round(mean_ch_width_m, 1),
    }

    # Print summary
    print(f'\n{"="*55}')
    print(f'YEAR {YEAR} — Morphometric Extraction')
    print(f'{"="*55}')
    print(f'\n── Area (km²) ───────────────────────────────────')
    print(f'  Braidplain total  : {BRAIDPLAIN_AREA_KM2:>8.2f} km²')
    print(f'  Active Channel    : {area_channel:>8.2f} km²  '
          f'({100*frac_channel:.1f}% of braidplain)')
    print(f'  Bare Bar          : {area_bare_bar:>8.2f} km²  '
          f'({100*frac_bare_bar:.1f}%)')
    print(f'  Vegetated Bar     : {area_veg_bar:>8.2f} km²  '
          f'({100*frac_veg_bar:.1f}%)')
    print(f'  Total Bar         : {area_total_bar:>8.2f} km²  '
          f'({100*frac_total_bar:.1f}%)')

    print(f'\n── Ratios ───────────────────────────────────────')
    print(f'  Veg fraction of bar  : {100*veg_of_bar:.1f}%')
    print(f'  Bare fraction of bar : {100*(1-veg_of_bar):.1f}%')
    print(f'  Mean channel width   : {mean_ch_width_m:.0f} m')
    print(f'  Bar/Channel ratio    : '
          f'{area_total_bar/max(area_channel,0.01):.3f}')

    print(f'\n── Years processed so far ───────────────────────')
    print(f'  {sorted(annual_metrics.keys())}')

    # Visualisation
    cmap   = mcolors.ListedColormap(CLASS_COLORS)
    norm_c = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)

    scale   = min(2000 / max(pred.shape), 1.0)
    pred_ds = pred[::max(1,int(1/scale)), ::max(1,int(1/scale))]

    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.imshow(pred_ds, cmap=cmap, norm=norm_c,
                   interpolation='nearest')
    ax.set_title(
        f'Attention U-Net — {YEAR} | '
        f'Channel={area_channel:.0f}km²  '
        f'TotalBar={area_total_bar:.0f}km²  '
        f'VegBar={area_veg_bar:.0f}km²\n'
        f'MeanChWidth={mean_ch_width_m:.0f}m  '
        f'VegFracOfBar={100*veg_of_bar:.1f}%',
        fontsize=10, fontweight='bold'
    )
    ax.axis('off')
    cbar = fig.colorbar(im, ax=ax, ticks=[0.5,1.5,2.5,3.5],
                        fraction=0.015, pad=0.02)
    cbar.ax.set_yticklabels(list(CLASS_NAMES.values()), fontsize=8)
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/morph_{YEAR}.png', dpi=150,
                bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Figure saved → morph_{YEAR}.png')

Assemble time series — run after all 8 years processed

In [ ]:
# ── ───────────────
if len(annual_metrics) < 8:
    print(f'Only {len(annual_metrics)} years processed so far.')
    print(f'Missing: '
          f'{set(["2019","2020","2021","2022","2023","2024","2025","2026"]) - set(annual_metrics.keys())}')
else:
    df_morph = (pd.DataFrame(annual_metrics)
                .T
                .sort_index()
                .reset_index(drop=True))
    df_morph['year'] = df_morph['year'].astype(int)
    df_morph = df_morph.set_index('year')

    print('Annual Morphometric Time Series:')
    print(df_morph.to_string())

    # Save
    df_morph.to_csv(f'{ROOT}/validation/annual_morphometrics.csv')
    print(f'\nSaved → annual_morphometrics.csv')

── Section 4B Cell 1: Generate 10km reach segments

Creates segments by slicing the braidplain along the reach direction.
Segments are ordered S01 (upstream/Tezpur) to S13 (downstream/Guwahati)
Output: reach_segments.gpkg — inspect in QGIS before proceeding

In [ ]:

REACH_LEN_KM  = 131
SEG_LEN_KM    = 10
N_SEGMENTS    = int(np.ceil(REACH_LEN_KM / SEG_LEN_KM))  # 14 to be safe
CRS           = 'EPSG:32646'

print(f'Reach length  : {REACH_LEN_KM} km')
print(f'Segment length: {SEG_LEN_KM} km')
print(f'N segments    : {N_SEGMENTS}')

# ── Step 1: Vectorise braidplain mask ─────────────────────────────────────
import glob
val_path = sorted(glob.glob(
    os.path.join(INF_DIR, 'inference_2025_validation_mask*.tif')
))[0]

print(f'\nVectorising braidplain mask...')
with rasterio.open(val_path) as src:
    bp_arr      = src.read(1)
    bp_transform= src.transform
    bp_crs      = src.crs

    # Vectorise braidplain (value > 0)
    shapes = list(features.shapes(
        (bp_arr > 0).astype(np.uint8),
        mask    = (bp_arr > 0).astype(np.uint8),
        transform= bp_transform
    ))

# Build braidplain polygon
bp_geoms = [Polygon(s['coordinates'][0]) for s, v in shapes if v == 1]
bp_union = unary_union(bp_geoms)
print(f'Braidplain polygon area: {bp_union.area/1e6:.1f} km²')

# ── Step 2: Get braidplain bounding box and main axis ─────────────────────
# The reach runs approximately WSW-ENE
# We slice perpendicular to the main reach axis
# Use the bounding box to define the slice direction

minx, miny, maxx, maxy = bp_union.bounds
print(f'\nBraidplain bounds (UTM):')
print(f'  X: {minx:.0f} to {maxx:.0f}  (range {(maxx-minx)/1000:.1f} km)')
print(f'  Y: {miny:.0f} to {maxy:.0f}  (range {(maxy-miny)/1000:.1f} km)')

# Determine dominant axis — the longer dimension is the along-reach direction
x_range = maxx - minx
y_range = maxy - miny
print(f'\nDominant axis: {"X (E-W)" if x_range > y_range else "Y (N-S)"}')

# ── Step 3: Create slicing strips along dominant axis ─────────────────────
# Slice the bounding box into N_SEGMENTS strips
# Each strip is a rectangle spanning the full cross-reach width
# Intersect each strip with braidplain polygon to get segment shape

SEG_LEN_M = SEG_LEN_KM * 1000  # 10,000 m

if x_range >= y_range:
    # Reach runs E-W — slice along X axis (Tezpur=East, Guwahati=West)
    # Bhomoraguri is upstream east, Pandu downstream west
    # So segments go from high X (east/Tezpur) to low X (west/Guwahati)
    slice_coords = np.linspace(maxx, minx, N_SEGMENTS + 1)

    segments = []
    for i in range(N_SEGMENTS):
        x_high = slice_coords[i]
        x_low  = slice_coords[i + 1]
        strip  = box(x_low, miny - 1000, x_high, maxy + 1000)
        seg    = bp_union.intersection(strip)

        if seg.is_empty or seg.area < 1e6:
            continue

        seg_id = f'S{i+1:02d}'
        segments.append({
            'seg_id'    : seg_id,
            'seg_num'   : i + 1,
            'x_upstream': round(x_high / 1000, 2),
            'x_dnstream': round(x_low  / 1000, 2),
            'area_km2'  : round(seg.area / 1e6, 2),
            'geometry'  : seg
        })
        print(f'  {seg_id}: area={seg.area/1e6:.1f} km²')

else:
    # Reach runs N-S — slice along Y axis
    slice_coords = np.linspace(miny, maxy, N_SEGMENTS + 1)

    segments = []
    for i in range(N_SEGMENTS):
        y_low  = slice_coords[i]
        y_high = slice_coords[i + 1]
        strip  = box(minx - 1000, y_low, maxx + 1000, y_high)
        seg    = bp_union.intersection(strip)

        if seg.is_empty or seg.area < 1e6:
            continue

        seg_id = f'S{i+1:02d}'
        segments.append({
            'seg_id'    : seg_id,
            'seg_num'   : i + 1,
            'y_upstream': round(y_low  / 1000, 2),
            'y_dnstream': round(y_high / 1000, 2),
            'area_km2'  : round(seg.area / 1e6, 2),
            'geometry'  : seg
        })
        print(f'  {seg_id}: area={seg.area/1e6:.1f} km²')

# ── Step 4: Build GeoDataFrame and save ───────────────────────────────────
gdf_segs = gpd.GeoDataFrame(segments, crs=CRS)

# Add approximate distance from upstream end
gdf_segs['dist_from_upstream_km'] = (gdf_segs['seg_num'] - 1) * SEG_LEN_KM

output_path = os.path.join(VAL_DIR, 'reach_segments.gpkg')
gdf_segs.to_file(output_path, driver='GPKG')

print(f'\n✓ Saved {len(gdf_segs)} segments → {output_path}')
print(f'\nSegment summary:')
print(gdf_segs[['seg_id','area_km2','dist_from_upstream_km']].to_string(index=False))
print(f'\nInspect in QGIS before proceeding to Cell 2.')
print(f'Load reach_segments.gpkg alongside your NIR false colour')
print(f'to verify segments are ordered correctly upstream to downstream.')

── Section 4B Cell 3: Per-segment morphometric extraction — 2025 test ────
After confirming 2025 looks correct, run Cell 4 for all years

In [ ]:
RES_KM2      = (10**2) / 1e6
REACH_LEN_KM = 131
CLASS_NAMES  = {0:'Background', 1:'Active Channel',
                2:'Active Bar',  3:'Vegetated Bar'}
CLASS_COLORS = ['#1A0400','#66CCFF','#E6F2FF','#66FFB3']

# ── Load segments ──────────────────────────────────────
gdf_segs = gpd.read_file(os.path.join(SEG_DIR, 'segments.shp'))
print(f'Segments: {len(gdf_segs)}')


def extract_segment_metrics(pred_path, gdf_segs, year):
    """
    Extract per-segment morphometric metrics from one annual prediction.
    Returns DataFrame with one row per segment.
    """
    results = []

    with rasterio.open(pred_path) as src:
        pred_crs = src.crs

        # Reproject segments to prediction CRS if needed
        segs = gdf_segs.to_crs(pred_crs) \
               if gdf_segs.crs != pred_crs else gdf_segs

        for _, row in segs.iterrows():
            seg_id = str(row['seg_id'])
            geom   = [mapping(row.geometry)]

            try:
                # Mask prediction to segment polygon
                arr, _ = rio_mask(src, geom,
                                  crop=True, nodata=0)
                arr = arr[0]   # (H, W)

            except Exception:
                # Segment outside raster extent
                results.append({'year': int(year), 'seg_id': seg_id,
                                 'channel_area_km2': np.nan,
                                 'bare_bar_area_km2': np.nan,
                                 'veg_bar_area_km2': np.nan,
                                 'total_bar_area_km2': np.nan,
                                 'frac_channel': np.nan,
                                 'frac_total_bar': np.nan,
                                 'frac_veg_bar': np.nan,
                                 'veg_of_bar': np.nan,
                                 'mean_ch_width_m': np.nan,
                                 'CCI': np.nan})
                continue

            # Count active pixels only (exclude background=0)
            n_channel  = int(np.sum(arr == 1))
            n_bare_bar = int(np.sum(arr == 2))
            n_veg_bar  = int(np.sum(arr == 3))
            n_total_bar= n_bare_bar + n_veg_bar
            n_active   = n_channel + n_total_bar

            if n_active == 0:
                results.append({'year': int(year), 'seg_id': seg_id,
                                 'channel_area_km2': 0,
                                 'bare_bar_area_km2': 0,
                                 'veg_bar_area_km2': 0,
                                 'total_bar_area_km2': 0,
                                 'frac_channel': np.nan,
                                 'frac_total_bar': np.nan,
                                 'frac_veg_bar': np.nan,
                                 'veg_of_bar': np.nan,
                                 'mean_ch_width_m': np.nan,
                                 'CCI': np.nan})
                continue

            # Areas
            area_channel   = n_channel   * RES_KM2
            area_bare_bar  = n_bare_bar  * RES_KM2
            area_veg_bar   = n_veg_bar   * RES_KM2
            area_total_bar = n_total_bar * RES_KM2

            # Fractions relative to active braidplain in segment
            frac_channel   = n_channel   / n_active
            frac_total_bar = n_total_bar / n_active
            frac_veg_bar   = n_veg_bar   / n_active
            veg_of_bar     = n_veg_bar    / max(n_total_bar, 1)

            # Segment reach length — proportional to segment area
            # Use segment active area / total active area × reach length
            seg_len_km = (n_active * RES_KM2) / \
                         (area_channel + area_total_bar) * \
                         (REACH_LEN_KM / len(gdf_segs))
            mean_ch_width_m = (area_channel / (REACH_LEN_KM / len(gdf_segs))) * 1000

            # Dynamic Braiding  Index [(1 - frac_channel) × frac_total_bar] / (1 + veg_of_bar)

            # DBI = [(1 - frac_channel) × (1 - veg_of_bar)] / √(Area of segment)
            # Higher = more complex braiding with more stable bars
            seg_area_km2 = n_active * RES_KM2
            DBI = ((1 - frac_channel) * frac_total_bar) / (1 + veg_of_bar)/ np.sqrt(max(seg_area_km2, 1e-6))

            results.append({
                'year'              : int(year),
                'seg_id'            : seg_id,
                'channel_area_km2'  : round(area_channel,   3),
                'bare_bar_area_km2' : round(area_bare_bar,  3),
                'veg_bar_area_km2'  : round(area_veg_bar,   3),
                'total_bar_area_km2': round(area_total_bar, 3),
                'frac_channel'      : round(frac_channel,   4),
                'frac_total_bar'    : round(frac_total_bar, 4),
                'frac_veg_bar'      : round(frac_veg_bar,   4),
                'veg_of_bar'        : round(veg_of_bar,     4),
                'mean_ch_width_m'   : round(mean_ch_width_m,1),
                'DBI'               : round(DBI,            4),
            })

    return pd.DataFrame(results)


# ── Test on 2025 ──────────────────────────────────────
pred_2025 = os.path.join(
    PRED_DIR, 'attention_unet_pred_2025_braidplain.tif'
)
df_2025 = extract_segment_metrics(pred_2025, gdf_segs, '2025')
df_2025['seg_id_num'] = df_2025['seg_id'].str.extract('(\d+)').astype(int)
df_2025 = df_2025.sort_values('seg_id_num').reset_index(drop=True)

print('\n── 2025 Per-Segment Metrics ─────────────────────────────────')
print(f"{"Seg":6s}  {"ChArea":>8s}  {"BarArea":>8s}  " \
      f"{"VegBar":>8s}  {"FracCh":>7s}  {"VegOfBar":>8s}  {"CCI":>7s}")
print('-'*65)
for _, row in df_2025.iterrows():
    print(f"{row["seg_id"]:6s}  " \
          f"{row["channel_area_km2"]:>8.2f}  " \
          f"{row["total_bar_area_km2"]:>8.2f}  " \
          f"{row["veg_bar_area_km2"]:>8.2f}  " \
          f"{row["frac_channel"]:>7.3f}  " \
          f"{row["veg_of_bar"]:>8.3f}  " \
          f"{row["DBI"]:>7.3f}")

print(f'\nReach totals:')
print(f'  Channel area   : {df_2025["channel_area_km2"].sum():.2f} km²')
print(f'  Total bar area : {df_2025["total_bar_area_km2"].sum():.2f} km²')
print(f'  Veg bar area   : {df_2025["veg_bar_area_km2"].sum():.2f} km²')
print(f'  Mean CCI       : {df_2025["DBI"].mean():.3f}')

# ── Spatial profile plot ────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('2025 Spatial Profile — Per-Segment Morphometrics',
             fontsize=12, fontweight='bold')

seg_labels = df_2025['seg_id'].tolist()
x          = np.arange(len(seg_labels))

# Area stacked bar
axes[0].bar(x, df_2025['channel_area_km2'],
            label='Channel', color='#66CCFF')
axes[0].bar(x, df_2025['bare_bar_area_km2'],
            bottom=df_2025['channel_area_km2'],
            label='Bare Bar', color='#E6F2FF')
axes[0].bar(x, df_2025['veg_bar_area_km2'],
            bottom=df_2025['channel_area_km2'] + df_2025['bare_bar_area_km2'],
            label='Veg Bar', color='#66FFB3')
axes[0].set_ylabel('Area (km²)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_title('Class Area per Segment')

# Fractions
axes[1].plot(x, df_2025['frac_channel'],
             'o-', color='#2196F3', linewidth=2, label='Channel fraction')
axes[1].plot(x, df_2025['frac_total_bar'],
             's-', color='#4CAF50', linewidth=2, label='Bar fraction')
axes[1].plot(x, df_2025['veg_of_bar'],
             '^--', color='#FF9800', linewidth=1.5,
             label='Veg fraction of bar')
axes[1].set_ylabel('Fraction')
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('Class Fractions per Segment')

# CCI
axes[2].bar(x, df_2025['DBI'], color='#9C27B0', alpha=0.8)
axes[2].set_ylabel('DBI')
axes[2].set_xticks(x)
axes[2].set_xticklabels(seg_labels, rotation=45, fontsize=8)
axes[2].set_xlabel('Segment (Upstream → Downstream)')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].set_title('Channel Complexity Index per Segment')

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/segment_profile_2025.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# ── Section 4B Cell 4: Full spatiotemporal extraction — all 8 years ───────
# Run only after confirming 2025 extraction looks correct

YEARS = ['2019','2020','2021','2022','2023','2024','2025','2026']

all_segments = []

for year in YEARS:
    pred_path = os.path.join(
        PRED_DIR, f'attention_unet_pred_{year}_braidplain.tif'
    )
    if not os.path.exists(pred_path):
        print(f'  MISSING: {year} — skipping')
        continue

    print(f'Processing {year}...', end=' ')
    df_yr = extract_segment_metrics(pred_path, gdf_segs, year)
    all_segments.append(df_yr)
    print(f'✓  mean CCI={df_yr["DBI"].mean():.3f}  '
          f'total bar={df_yr["total_bar_area_km2"].sum():.1f}km²')

# Combine all years
df_spatio = pd.concat(all_segments, ignore_index=True)
df_spatio = df_spatio.sort_values(['year','seg_id']).reset_index(drop=True)

print(f'\nSpatiotemporal dataframe: {df_spatio.shape}')
print(f'Years: {sorted(df_spatio["year"].unique())}')
print(f'Segments: {sorted(df_spatio["seg_id"].unique())}')

# Save
csv_path = os.path.join(VAL_DIR, 'spatiotemporal_morphometrics.csv')
df_spatio.to_csv(csv_path, index=False)
print(f'Saved → {csv_path}')

# ── CCI spatiotemporal heatmap ────────────────────────────────────────────
pivot_CCI = df_spatio.pivot(
    index='year', columns='seg_id', values='DBI'
)
pivot_CCI = pivot_CCI.reindex(
    sorted(pivot_CCI.columns, key=lambda x: int(x.replace('S',''))),
    axis=1
)

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(pivot_CCI.values, aspect='auto',
               cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(len(pivot_CCI.columns)))
ax.set_xticklabels(pivot_CCI.columns, fontsize=10)
ax.set_yticks(range(len(pivot_CCI.index)))
ax.set_yticklabels(pivot_CCI.index, fontsize=9)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Segment (Upstream → Downstream)', fontsize=11)
ax.set_title(
    'Channel Complexity Index — Spatiotemporal Heatmap\n'
    'Tezpur (S01) → Guwahati (S15) | 2019-2026',
    fontsize=12, fontweight='bold'
)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('CCI', fontsize=10)

# Annotate cells with values
for i in range(len(pivot_CCI.index)):
    for j in range(len(pivot_CCI.columns)):
        val = pivot_CCI.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}',
                    ha='center', va='center',
                    fontsize=7, color='black')

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/CCI_spatiotemporal_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('CCI heatmap saved.')

# ── Total bar area heatmap ────────────────────────────────────────────────
pivot_bar = df_spatio.pivot(
    index='year', columns='seg_id', values='total_bar_area_km2'
)
pivot_bar = pivot_bar.reindex(
    sorted(pivot_bar.columns, key=lambda x: int(x.replace('S',''))),
    axis=1
)

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(pivot_bar.values, aspect='auto',
               cmap='Blues_r', interpolation='nearest')
ax.set_xticks(range(len(pivot_bar.columns)))
ax.set_xticklabels(pivot_bar.columns, fontsize=10)
ax.set_yticks(range(len(pivot_bar.index)))
ax.set_yticklabels(pivot_bar.index, fontsize=9)
ax.set_xlabel('Segment (Upstream → Downstream)')
ax.set_ylabel('Year')
ax.set_title(
    'Total Bar Area (km²) — Spatiotemporal Heatmap\n'
    'Tezpur (S01) → Guwahati (S15) | 2019-2026',
    fontsize=12, fontweight='bold'
)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Total Bar Area (km²)', fontsize=10)

for i in range(len(pivot_bar.index)):
    for j in range(len(pivot_bar.columns)):
        val = pivot_bar.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.1f}',
                    ha='center', va='center',
                    fontsize=7, color='black')

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/bar_area_spatiotemporal_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Bar area heatmap saved.')